# eQTL / GWAS-catalog overlap by activity

**Reviewer #4** asked whether within-human variation in these cCREs is more likely to generate eQTLs, and whether the genes/regions are enriched for GWAS associations. **Response (this part):** we tested eQTL overlap (cartilage eQTLs, catalog SNPs, GTEx distances) and GWAS-catalog overlap, stratified by MPRA activity.

In [ ]:
import pandas as pd
from Bio import SeqIO
import sys
import os
from ast import literal_eval
import numpy as np
import pybedtools
from pybedtools import BedTool
from matplotlib.colors import LinearSegmentedColormap
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy import stats
from scipy.stats import gaussian_kde
from scipy.stats import pearsonr
from scipy.stats import spearmanr
import itertools
import re
import ast
import math

# --- Shared helpers: const.py lives in analyses/common ---
NB_DIR = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(NB_DIR, '..', 'common')))
import const
from const import pos_active_ctrl_color, neg_active_ctrl_color, highlight_color, custom_cmap
from const import set_equal_plot_limits, plot_color_pallete, custom_cmap_bolder, FONT_SIZE_small
const.set_plot_style()

# --- Output directory (EDIT): where plots/tables are written ---
output_path = '.'

# --- External liftOver tool (EDIT): used to convert genome coordinates ---
LIFTOVER_DIR = '/home/labs/davidgo/Collaboration/Lab_Tools/liftOver/no_GUI'
os.chdir(LIFTOVER_DIR)
from liftOver import initializer as initializer
from liftOver import liftOver_df as liftOver
os.chdir(NB_DIR)


## eQTL

### Explore eQTL effect size distribution

### Load cartilage eQTL data

In [ ]:
high_grade_cartilage_eQTLs = pd.read_csv("/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Expression/Cartilage/Human/cartilage_eQTLs_for_healthy_and_OA_individuals/processed_data/HighGradeCartilage_significant_eQTLs.txt", sep="\t")
low_grade_cartilage_eQTLs = pd.read_csv("/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Expression/Cartilage/Human/cartilage_eQTLs_for_healthy_and_OA_individuals/processed_data/LowGradeCartilage_significant_eQTLs.txt", sep="\t")

high_grade_cartilage_eQTLs["origin"] = "high_grade"
low_grade_cartilage_eQTLs["origin"] = "low_grade"

cartilage_eQTLs = pd.concat(
    [high_grade_cartilage_eQTLs, low_grade_cartilage_eQTLs],
    ignore_index=True
)

# filter eQTLs
#cartilage_eQTLs = cartilage_eQTLs[
#    (cartilage_eQTLs["pval"] < 0.00001) &
#    (cartilage_eQTLs["slope"].abs() > 0.5)
#].copy()

cartilage_eQTLs["chromosome"] = cartilage_eQTLs["genotype_id"].apply(lambda x: x.split(":")[0])
cartilage_eQTLs["position"] = cartilage_eQTLs["genotype_id"].apply(lambda x: int(x.split(":")[1]))

### Load and filter oligo library

In [ ]:
# Read only the specified columns from the CSV file
oligos = pd.read_csv(f'/home/labs/davidgo/Collaboration/humanMPRA/top_candidates/chondrocytes/humanMPRA_annotations_v3.csv', 
                     #usecols=range(0, 25), 
                     header=0)
print('Number of oligos in hMPRA:', len(oligos))
oligos = oligos.drop_duplicates(subset=["oligo"], keep = "first") #There are several HH oligos which are duplicated
print('Number of oligos in hMPRA without duplicates:', len(oligos))

min_DNA_counts = 50

oligos = oligos[(oligos['DNA_counts_raw_derived']>min_DNA_counts) &
                               (oligos['DNA_counts_raw_ancestral']>min_DNA_counts)]
print('Number of oligos in hMPRA after filtering for at least 50 DNA counts in both archaic and derived:', len(oligos))


ATAC_seq_filter = False
ATAC_seq_distance = 5000

if ATAC_seq_filter:
    print('Number of oligos in hMPRA before filtering for being in an ATAC-seq peak:', len(oligos))
    oligos["ATAC_seq_bool"] = oligos["ATACseq_peaks_fetal_chondrocytes"].abs() < ATAC_seq_distance
    oligos = oligos[oligos["ATAC_seq_bool"] == True]
    print('Number of oligos in hMPRA after filtering for being in an ATAC-seq peak:', len(oligos))


In [ ]:
# Parameter: which eQTL source to use for overlap analysis
use_gtex_eqtls = True  # Set to True to analyze GTEx eQTLs instead of chondrocyte

### Inspect oligo dataframe

## eQTL Overlap Analysis: diff_active vs active vs non_active

### Define eQTL overlap function

In [ ]:
def check_eqtl_overlap(oligos_df, eqtl_df,
                       oligo_chr_col="chromosome",
                       oligo_start_col="start",
                       oligo_end_col="end",
                       oligo_activity_col="differential_activity",
                       eqtl_chr_col="chromosome",
                       eqtl_pos_col="position"):
    """
    Check for binary overlap between oligo intervals and eQTL positions.
    
    Parameters:
    -----------
    oligos_df : DataFrame
        Oligo dataframe with coordinates and activity labels
    eqtl_df : DataFrame
        eQTL dataframe with SNP positions
    oligo_chr_col : str
        Chromosome column name in oligos_df
    oligo_start_col : str
        Start coordinate column in oligos_df
    oligo_end_col : str
        End coordinate column in oligos_df
    oligo_activity_col : str
        Differential activity column in oligos_df
    eqtl_chr_col : str
        Chromosome column name in eqtl_df
    eqtl_pos_col : str
        Position column in eqtl_df
    
    Returns:
    --------
    DataFrame with columns:
        - has_eqtl_overlap: 1 if any eQTL overlaps with oligo interval, 0 otherwise
        - activity_group: Categorical assignment based on differential_activity
    """
    
    # Prepare oligo data
    oligos_overlap = oligos_df.copy()
    cols_needed = [oligo_chr_col, oligo_start_col, oligo_end_col, oligo_activity_col]
    missing = [c for c in cols_needed if c not in oligos_overlap.columns]
    if missing:
        raise ValueError(f"Missing required oligos columns: {missing}")
    
    # Keep only necessary columns
    oligos_overlap = oligos_overlap[[oligo_chr_col, oligo_start_col, oligo_end_col, oligo_activity_col]].copy()
    
    # Convert coordinates to numeric
    oligos_overlap[oligo_start_col] = pd.to_numeric(oligos_overlap[oligo_start_col], errors="coerce")
    oligos_overlap[oligo_end_col] = pd.to_numeric(oligos_overlap[oligo_end_col], errors="coerce")
    
    # Drop rows with missing coordinates
    oligos_overlap = oligos_overlap.dropna(subset=[oligo_chr_col, oligo_start_col, oligo_end_col]).copy()
    oligos_overlap[oligo_start_col] = oligos_overlap[oligo_start_col].astype(int)
    oligos_overlap[oligo_end_col] = oligos_overlap[oligo_end_col].astype(int)
    
    # Clean chromosome names
    oligos_overlap["chrom_clean"] = oligos_overlap[oligo_chr_col].astype(str).str.replace("^chr", "", regex=True)
    
    # Prepare eQTL data
    eqtl_proc = eqtl_df.copy()
    missing_eqtl = [c for c in [eqtl_chr_col, eqtl_pos_col] if c not in eqtl_proc.columns]
    if missing_eqtl:
        raise ValueError(f"Missing required eQTL columns: {missing_eqtl}")
    
    eqtl_proc = eqtl_proc[[eqtl_chr_col, eqtl_pos_col]].dropna().copy()
    eqtl_proc[eqtl_pos_col] = pd.to_numeric(eqtl_proc[eqtl_pos_col], errors="coerce")
    eqtl_proc = eqtl_proc.dropna(subset=[eqtl_pos_col]).copy()
    eqtl_proc[eqtl_pos_col] = eqtl_proc[eqtl_pos_col].astype(int)
    eqtl_proc["chrom_clean"] = eqtl_proc[eqtl_chr_col].astype(str).str.replace("^chr", "", regex=True)
    
    # Check for overlaps
    has_overlap = []
    for chrom, grp in oligos_overlap.groupby("chrom_clean"):
        eqtl_pos = np.sort(eqtl_proc.loc[eqtl_proc["chrom_clean"] == chrom, eqtl_pos_col].values)
        
        starts = grp[oligo_start_col].to_numpy()
        ends = grp[oligo_end_col].to_numpy()
        
        if len(eqtl_pos) == 0:
            # No eQTLs on this chromosome
            overlap_flags = np.zeros(len(starts), dtype=int)
        else:
            overlap_flags = np.zeros(len(starts), dtype=int)
            
            for i in range(len(starts)):
                start = starts[i]
                end = ends[i]
                
                # Check if any eQTL falls within [start, end]
                if np.any((eqtl_pos >= start) & (eqtl_pos <= end)):
                    overlap_flags[i] = 1
        
        out = grp.copy()
        out["has_eqtl_overlap"] = overlap_flags
        has_overlap.append(out)
    
    result = pd.concat(has_overlap, ignore_index=True)
    
    # Classify activity groups based on differential_activity column
    # TRUE/1 = diff_active, FALSE/0 = active_not_diff, NaN = not_active
    status_map = {
        True: "diff_active",
        False: "active_not_diff",
        "TRUE": "diff_active",
        "FALSE": "active_not_diff",
        "True": "diff_active",
        "False": "active_not_diff",
        "true": "diff_active",
        "false": "active_not_diff",
        1: "diff_active",
        0: "active_not_diff"
    }
    
    result["activity_group"] = result[oligo_activity_col].map(status_map)
    result.loc[result["activity_group"].isna(), "activity_group"] = "not_active"
    
    return result[["chrom_clean", oligo_start_col, oligo_end_col, "has_eqtl_overlap", "activity_group", oligo_activity_col]]


### Define contingency table and visualization functions

In [ ]:
def compute_contingency_tables(overlap_df, bonferroni_correction=False):
    """
    Compute three pairwise contingency tables and perform chi-square tests.
    
    Parameters:
    -----------
    overlap_df : DataFrame
        Output from check_eqtl_overlap() with columns has_eqtl_overlap and activity_group
    bonferroni_correction : bool
        If True, adjust p-value threshold for multiple testing (α' = 0.05/3)
    
    Returns:
    --------
    Dictionary with three comparison results, each containing:
        - contingency_table: 2x2 contingency table
        - chi2_statistic: Chi-square test statistic
        - p_value: Chi-square test p-value
        - counts: Raw counts by group
        - percentages: Row-normalized percentages
    """
    
    results = {}
    comparison_names = [
        "diff_active vs active_not_diff",
        "diff_active vs not_active",
        "all_active vs not_active"
    ]
    
    # Comparison 1: diff_active vs active_not_diff
    comp1_df = overlap_df[overlap_df["activity_group"].isin(["diff_active", "active_not_diff"])].copy()
    if len(comp1_df) > 0:
        contingency_1 = pd.crosstab(comp1_df["activity_group"], comp1_df["has_eqtl_overlap"])
        chi2_1, p_val_1, dof_1, expected_1 = stats.chi2_contingency(contingency_1)
        
        results[comparison_names[0]] = {
            "contingency_table": contingency_1,
            "chi2_statistic": chi2_1,
            "p_value": p_val_1,
            "dof": dof_1,
            "n_samples": len(comp1_df),
            "counts": comp1_df["activity_group"].value_counts().to_dict()
        }
    
    # Comparison 2: diff_active vs not_active
    comp2_df = overlap_df[overlap_df["activity_group"].isin(["diff_active", "not_active"])].copy()
    if len(comp2_df) > 0:
        contingency_2 = pd.crosstab(comp2_df["activity_group"], comp2_df["has_eqtl_overlap"])
        chi2_2, p_val_2, dof_2, expected_2 = stats.chi2_contingency(contingency_2)
        
        results[comparison_names[1]] = {
            "contingency_table": contingency_2,
            "chi2_statistic": chi2_2,
            "p_value": p_val_2,
            "dof": dof_2,
            "n_samples": len(comp2_df),
            "counts": comp2_df["activity_group"].value_counts().to_dict()
        }
    
    # Comparison 3: all_active (diff + not_diff) vs not_active
    comp3_df = overlap_df.copy()
    comp3_df["compare_group"] = comp3_df["activity_group"].apply(
        lambda x: "all_active" if x in ["diff_active", "active_not_diff"] else "not_active"
    )
    if len(comp3_df) > 0:
        contingency_3 = pd.crosstab(comp3_df["compare_group"], comp3_df["has_eqtl_overlap"])
        chi2_3, p_val_3, dof_3, expected_3 = stats.chi2_contingency(contingency_3)
        
        results[comparison_names[2]] = {
            "contingency_table": contingency_3,
            "chi2_statistic": chi2_3,
            "p_value": p_val_3,
            "dof": dof_3,
            "n_samples": len(comp3_df),
            "counts": comp3_df["compare_group"].value_counts().to_dict()
        }
    
    # Print summary statistics
    alpha = 0.05 / 3 if bonferroni_correction else 0.05
    print("="*80)
    print("Chi-Square Test: Contingency Analysis of eQTL Overlap")
    print("="*80)
    if bonferroni_correction:
        print(f"Bonferroni-corrected significance level: α' = {alpha:.4f}\n")
    else:
        print(f"Significance level: α = {alpha:.4f}\n")
    
    for comp_name, result in results.items():
        print(f"\nComparison: {comp_name}")
        print(f"  Total oligos: {result['n_samples']}")
        print(f"  Groups: {list(result['counts'].keys())}")
        print(f"  Contingency Table:")
        print(result["contingency_table"].to_string())
        print(f"\n  Chi-square statistic: {result['chi2_statistic']:.4f}")
        print(f"  P-value: {result['p_value']:.4e}")
        
        if result['p_value'] < alpha:
            print(f"  ✓ SIGNIFICANT (p < {alpha:.4f}): eQTL overlap differs between groups")
        else:
            print(f"  ✗ Not significant (p >= {alpha:.4f}): No evidence of difference")
    
    return results


def plot_contingency_heatmaps(contingency_results, figsize=(16, 6), output_path=None):
    """
    Create heatmaps for contingency tables with raw counts and row-normalized percentages.
    
    Parameters:
    -----------
    contingency_results : dict
        Output from compute_contingency_tables()
    figsize : tuple
        Figure size for the combined plots
    output_path : str
        Optional path to save the figure
    
    Returns:
    --------
    None (displays plots)
    """
    
    y_label_map = {
        "diff_active":     "Diff. active",
        "active_not_diff": "Active (not diff.)",
        "not_active":      "Not active",
        "all_active":      "All active",
    }
    x_label_map = {0: "No eQTL", 1: "Has eQTL"}

    n_comparisons = len(contingency_results)
    fig, axes = plt.subplots(1, n_comparisons, figsize=figsize)
    
    if n_comparisons == 1:
        axes = [axes]
    
    for idx, (comp_name, result) in enumerate(contingency_results.items()):
        contingency = result["contingency_table"]
        chi2 = result["chi2_statistic"]
        pval = result["p_value"]
        
        # Compute row-normalized percentages
        contingency_pct = contingency.div(contingency.sum(axis=1), axis=0) * 100
        
        ax = axes[idx]
        
        sns.heatmap(
            contingency_pct,
            annot=False,
            cmap="YlOrRd",
            cbar_kws={"label": "% within group"},
            ax=ax,
            vmin=0,
            vmax=100,
            linewidths=0.5,
            linecolor="white",
        )
        
        # Annotate each cell with count and percentage, centered
        for i in range(contingency.shape[0]):
            for j in range(contingency.shape[1]):
                count = contingency.iloc[i, j]
                pct = contingency_pct.iloc[i, j]
                ax.text(j + 0.5, i + 0.5, f"{int(count):,}\n({pct:.1f}%)",
                        ha="center", va="center", fontsize=9,
                        color="black", fontweight="bold")
        
        ax.set_title(f"{comp_name}\nχ² = {chi2:.3f}, p = {pval:.2e}",
                     fontsize=11, fontweight="bold", pad=10)
        ax.set_xlabel("eQTL Overlap", labelpad=10)
        ax.set_ylabel("Oligo Category", labelpad=8)
        
        ax.set_xticklabels(
            [x_label_map.get(c, str(c)) for c in contingency.columns],
            rotation=45, ha="right", fontsize=10
        )
        ax.set_yticklabels(
            [y_label_map.get(r, str(r)) for r in contingency.index],
            rotation=0, fontsize=10
        )
    
    fig.tight_layout(pad=2.0)
    
    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches="tight")
        print(f"Figure saved to: {output_path}")
    
    plt.show()


### Run eQTL overlap analysis

In [ ]:
# Select eQTL source
eqtls_to_use = gtex_eQTLs_sampled if use_gtex_eqtls else cartilage_eQTLs
eqtl_source_name = "GTEx (subsampled)" if use_gtex_eqtls else "Chondrocyte"

# Execute eQTL overlap analysis with three pairwise comparisons
print("="*80)
print("eQTL Overlap Analysis: Binary Overlap (Yes/No) Across Activity Groups")
print(f"eQTL Source: {eqtl_source_name}")
print("="*80)
print(f"\nTotal {eqtl_source_name} eQTLs: {len(eqtls_to_use)}")
print(f"Total oligos: {len(oligos)}")

# Step 1: Check for eQTL overlaps
print("\n[Step 1] Computing eQTL overlaps...")
overlap_results = check_eqtl_overlap(
    oligos,
    eqtls_to_use,
    oligo_chr_col="chromosome",
    oligo_start_col="start",
    oligo_end_col="end",
    oligo_activity_col="differential_activity",
    eqtl_chr_col="chromosome",
    eqtl_pos_col="position"
)

print(f"Oligos with valid coordinates: {len(overlap_results)}")
print(f"\nActivity group distribution:")
print(overlap_results["activity_group"].value_counts())
print(f"\neQTL overlap distribution:")
print(overlap_results["has_eqtl_overlap"].value_counts())

# Step 2: Compute contingency tables and perform chi-square tests
print("\n[Step 2] Computing contingency tables and chi-square tests...")
contingency_results = compute_contingency_tables(overlap_results, bonferroni_correction=False)

# Step 3: Visualize as heatmaps
print("\n[Step 3] Creating heatmap visualizations...")
plot_contingency_heatmaps(
    contingency_results,
    figsize=(15, 5),
    output_path=f"{output_path}/eQTL_overlap_contingency_heatmaps_{eqtl_source_name.replace(' ', '_').lower()}.png"
)

print("\nAnalysis complete!")

## Catalog SNP Analysis: eQTL Overlap Stratified by Oligo Activity

### Define catalog SNP classification and eQTL matching functions

In [ ]:
def load_and_classify_catalog_snps(catalog_path, oligos_df,
                                   oligo_chr_col="chromosome",
                                   oligo_start_col="start",
                                   oligo_end_col="end",
                                   oligo_activity_col="differential_activity",
                                   catalog_sample_fraction=1.0,
                                   subsample_outside_snps=True,
                                   outside_snp_fraction=0.1,
                                   random_seed=42):
    """
    Load SNP catalog and classify each SNP by whether it overlaps with oligo groups.
    Subsamples SNPs outside all oligos to speed up computation.
    """
    
    # Load catalog
    print(f"Loading catalog from: {catalog_path}")
    catalog = pd.read_csv(
        catalog_path,
        sep="\t",
        header=None,
        names=["chromosome", "start", "end", "variant_type", "extra"]
    )
    print(f"Loaded {len(catalog)} SNPs from catalog")
    
    # Sample the catalog upfront before any expensive computation
    if catalog_sample_fraction < 1.0:
        catalog = catalog.sample(frac=catalog_sample_fraction, random_state=random_seed).reset_index(drop=True)
        print(f"Subsampled to {len(catalog)} SNPs ({catalog_sample_fraction*100:.0f}% of catalog)")
    
    # Prepare oligos with activity classification
    oligos_proc = oligos_df[[oligo_chr_col, oligo_start_col, oligo_end_col, oligo_activity_col]].copy()
    oligos_proc = oligos_proc.dropna(subset=[oligo_chr_col, oligo_start_col, oligo_end_col]).copy()
    oligos_proc[oligo_start_col] = pd.to_numeric(oligos_proc[oligo_start_col], errors="coerce")
    oligos_proc[oligo_end_col] = pd.to_numeric(oligos_proc[oligo_end_col], errors="coerce")
    oligos_proc = oligos_proc.dropna(subset=[oligo_start_col, oligo_end_col]).copy()
    oligos_proc[oligo_start_col] = oligos_proc[oligo_start_col].astype(int)
    oligos_proc[oligo_end_col] = oligos_proc[oligo_end_col].astype(int)
    
    # Classify oligo activity
    status_map = {
        True: "diff_active", False: "active_not_diff",
        "TRUE": "diff_active", "FALSE": "active_not_diff",
        "True": "diff_active", "False": "active_not_diff",
        "true": "diff_active", "false": "active_not_diff",
        1: "diff_active", 0: "active_not_diff"
    }
    oligos_proc["activity"] = oligos_proc[oligo_activity_col].map(status_map)
    oligos_proc.loc[oligos_proc["activity"].isna(), "activity"] = "not_active"
    
    # Clean chromosome names
    oligos_proc["chrom_clean"] = oligos_proc[oligo_chr_col].astype(str).str.replace("^chr", "", regex=True)
    catalog["chrom_clean"] = catalog["chromosome"].astype(str).str.replace("^chr", "", regex=True)
    
    print("Building interval index for efficient overlap checking...")
    
    # Build interval dictionaries per chromosome and activity group.
    # NOTE: starts and ends must NOT be sorted independently — that would unpair them.
    intervals_any = {}
    intervals_active = {}
    intervals_diff_active = {}
    
    for chrom in oligos_proc["chrom_clean"].unique():
        chrom_oligos = oligos_proc[oligos_proc["chrom_clean"] == chrom]
        
        starts_any = chrom_oligos[oligo_start_col].values
        ends_any = chrom_oligos[oligo_end_col].values
        
        if len(starts_any) > 0:
            intervals_any[chrom] = (starts_any, ends_any)
        
        active = chrom_oligos[chrom_oligos["activity"].isin(["diff_active", "active_not_diff"])]
        if len(active) > 0:
            intervals_active[chrom] = (active[oligo_start_col].values, active[oligo_end_col].values)
        
        diff = chrom_oligos[chrom_oligos["activity"] == "diff_active"]
        if len(diff) > 0:
            intervals_diff_active[chrom] = (diff[oligo_start_col].values, diff[oligo_end_col].values)
    
    def check_overlap_vectorized(snp_positions, snp_chroms, intervals_dict):
        """For each SNP, check if it falls inside any interval on its chromosome."""
        overlaps = np.zeros(len(snp_positions), dtype=int)
        
        for chrom in np.unique(snp_chroms):
            if chrom not in intervals_dict:
                continue
            
            chrom_mask = snp_chroms == chrom
            chrom_snp_pos = snp_positions[chrom_mask]
            starts, ends = intervals_dict[chrom]
            
            chrom_overlaps = np.zeros(chrom_snp_pos.shape, dtype=int)
            for j, snp_pos in enumerate(chrom_snp_pos):
                if np.any((starts <= snp_pos) & (ends >= snp_pos)):
                    chrom_overlaps[j] = 1
            
            overlaps[chrom_mask] = chrom_overlaps
        
        return overlaps
    
    print("Checking SNP overlaps with oligo intervals...")
    snp_positions = catalog["start"].values
    snp_chroms = catalog["chrom_clean"].values
    
    catalog["overlaps_any_oligo"] = check_overlap_vectorized(snp_positions, snp_chroms, intervals_any)
    
    # Subsample SNPs outside all oligos if requested
    if subsample_outside_snps:
        print(f"\nSubsampling SNPs outside all oligos...")
        outside_mask = catalog["overlaps_any_oligo"] == 0
        inside_mask = catalog["overlaps_any_oligo"] == 1
        
        n_outside = outside_mask.sum()
        n_inside = inside_mask.sum()
        
        print(f"  SNPs outside all oligos: {n_outside:,}")
        print(f"  SNPs inside any oligo: {n_inside:,}")
        
        n_keep = max(int(n_outside * outside_snp_fraction), 1000)
        outside_indices = np.where(outside_mask)[0]
        np.random.seed(random_seed)
        keep_indices = np.random.choice(outside_indices, size=min(n_keep, len(outside_indices)), replace=False)
        inside_indices = np.where(inside_mask)[0]
        
        keep_all_indices = np.sort(np.concatenate([inside_indices, keep_indices]))
        catalog = catalog.iloc[keep_all_indices].reset_index(drop=True)
        
        print(f"  Keeping {len(keep_indices):,} subsampled outside SNPs ({outside_snp_fraction*100:.1f}%)")
        print(f"  Total SNPs after subsampling: {len(catalog):,}")
    
    print("Checking overlaps with active and diff_active oligos...")
    snp_positions = catalog["start"].values
    snp_chroms = catalog["chrom_clean"].values
    
    catalog["overlaps_active_oligo"] = check_overlap_vectorized(snp_positions, snp_chroms, intervals_active)
    catalog["overlaps_diff_active_oligo"] = check_overlap_vectorized(snp_positions, snp_chroms, intervals_diff_active)
    
    def classify_snp(row):
        if row["overlaps_diff_active_oligo"] == 1:
            return "in_diff_active"
        elif row["overlaps_active_oligo"] == 1:
            return "in_active"
        elif row["overlaps_any_oligo"] == 1:
            return "in_any_oligo"
        else:
            return "outside"
    
    print("Classifying SNPs into groups...")
    catalog["snp_group"] = catalog.apply(classify_snp, axis=1)
    
    return catalog


def check_snp_eqtl_overlap(catalog_df, eqtl_df,
                            catalog_pos_col="start",
                            catalog_chr_col="chromosome",
                            eqtl_chr_col="chromosome",
                            eqtl_pos_col="position",
                            window=270):
    """
    Check if any eQTL position falls within `window` bp of each catalog SNP.
    
    The default window of 270 bp (MPRA oligo size) means a catalog SNP "has eQTL overlap"
    if an eQTL falls anywhere in the same oligo-sized window — which is the biologically
    meaningful question for this MPRA analysis.
    
    Catalog positions are BED 0-based; eQTL positions are 1-based. The function converts
    catalog start to 1-based before the window comparison.
    
    Adds has_eqtl_match column (1 = match, 0 = no match).
    """
    catalog = catalog_df.copy()
    
    catalog["chrom_clean_snp"] = catalog[catalog_chr_col].astype(str).str.replace("^chr", "", regex=True)
    
    eqtl_proc = eqtl_df[[eqtl_chr_col, eqtl_pos_col]].dropna().copy()
    eqtl_proc[eqtl_pos_col] = pd.to_numeric(eqtl_proc[eqtl_pos_col], errors="coerce")
    eqtl_proc = eqtl_proc.dropna(subset=[eqtl_pos_col]).copy()
    eqtl_proc[eqtl_pos_col] = eqtl_proc[eqtl_pos_col].astype(int)
    eqtl_proc["chrom_clean_eqtl"] = eqtl_proc[eqtl_chr_col].astype(str).str.replace("^chr", "", regex=True)
    
    # Build sorted eQTL position arrays per chromosome for fast binary-search window queries
    eqtl_arrays = {}
    for chrom, grp in eqtl_proc.groupby("chrom_clean_eqtl"):
        eqtl_arrays[chrom] = np.sort(grp[eqtl_pos_col].values)
    
    has_match = np.zeros(len(catalog), dtype=int)
    
    for chrom in catalog["chrom_clean_snp"].unique():
        if chrom not in eqtl_arrays:
            continue
        
        chrom_mask = catalog["chrom_clean_snp"].values == chrom
        idx = np.where(chrom_mask)[0]
        # BED start is 0-based; eQTL positions are 1-based — convert for comparison
        snp_pos_1based = catalog.iloc[idx][catalog_pos_col].values + 1
        eqtl_pos_arr = eqtl_arrays[chrom]
        
        chrom_matches = np.zeros(len(idx), dtype=int)
        for j, snp_pos in enumerate(snp_pos_1based):
            lo = snp_pos - window
            hi = snp_pos + window
            # Binary search: find leftmost eQTL >= lo, then check if it's also <= hi
            i_lo = np.searchsorted(eqtl_pos_arr, lo)
            if i_lo < len(eqtl_pos_arr) and eqtl_pos_arr[i_lo] <= hi:
                chrom_matches[j] = 1
        
        has_match[idx] = chrom_matches
    
    catalog["has_eqtl_match"] = has_match
    catalog.drop(columns=["chrom_clean_snp"], inplace=True)
    
    return catalog


### Define catalog contingency table and heatmap functions

In [ ]:
def compute_catalog_contingency_tables(snp_catalog_df, bonferroni_correction=False):
    """
    Compute three pairwise contingency tables for catalog SNPs stratified by oligo group.
    
    Comparisons:
    1. SNPs in any oligo vs SNPs outside all oligos (by eQTL overlap)
    2. SNPs in active oligos vs SNPs outside all oligos (by eQTL overlap)
    3. SNPs in diff_active oligos vs SNPs outside all oligos (by eQTL overlap)
    
    Parameters:
    -----------
    snp_catalog_df : DataFrame
        Output from check_snp_eqtl_overlap() with columns snp_group and has_eqtl_match
    bonferroni_correction : bool
        If True, adjust p-value threshold (α' = 0.05/3)
    
    Returns:
    --------
    Dictionary with three comparison results
    """
    
    results = {}
    comparison_configs = [
        {
            "name": "SNPs in any oligo vs outside",
            "groups": ["in_any_oligo", "outside"],
            "labels": ["In any oligo", "Outside all oligos"]
        },
        {
            "name": "SNPs in active oligos vs outside",
            "groups": ["in_active", "outside"],
            "labels": ["In active oligos", "Outside all oligos"]
        },
        {
            "name": "SNPs in diff_active oligos vs outside",
            "groups": ["in_diff_active", "outside"],
            "labels": ["In diff_active oligos", "Outside all oligos"]
        }
    ]
    
    for config in comparison_configs:
        comp_name = config["name"]
        groups_to_compare = config["groups"]
        group_labels = config["labels"]
        
        # Filter to relevant groups
        comp_df = snp_catalog_df[snp_catalog_df["snp_group"].isin(groups_to_compare)].copy()
        
        if len(comp_df) > 0:
            # Create contingency table
            contingency = pd.crosstab(comp_df["snp_group"], comp_df["has_eqtl_match"])
            chi2, p_val, dof, expected = stats.chi2_contingency(contingency)
            
            # Store results
            results[comp_name] = {
                "contingency_table": contingency,
                "chi2_statistic": chi2,
                "p_value": p_val,
                "dof": dof,
                "n_samples": len(comp_df),
                "counts_by_group": comp_df["snp_group"].value_counts().to_dict(),
                "labels": group_labels
            }
    
    # Print summary
    alpha = 0.05 / 3 if bonferroni_correction else 0.05
    print("="*80)
    print("Chi-Square Test: Catalog SNPs eQTL Overlap Analysis")
    print("="*80)
    if bonferroni_correction:
        print(f"Bonferroni-corrected significance level: α' = {alpha:.4f}\n")
    else:
        print(f"Significance level: α = {alpha:.4f}\n")
    
    for comp_name, result in results.items():
        print(f"\nComparison: {comp_name}")
        print(f"  Total SNPs: {result['n_samples']}")
        print(f"  SNP group counts: {result['counts_by_group']}")
        print(f"  Contingency Table:")
        print(result["contingency_table"].to_string())
        print(f"\n  Chi-square statistic: {result['chi2_statistic']:.4f}")
        print(f"  P-value: {result['p_value']:.4e}")
        
        if result['p_value'] < alpha:
            print(f"  ✓ SIGNIFICANT (p < {alpha:.4f}): eQTL match rate differs between groups")
        else:
            print(f"  ✗ Not significant (p >= {alpha:.4f}): No evidence of difference")
    
    return results


def plot_catalog_contingency_heatmaps(contingency_results, figsize=(16, 6), output_path=None):
    """
    Create heatmaps for catalog SNP contingency tables.
    
    Parameters:
    -----------
    contingency_results : dict
        Output from compute_catalog_contingency_tables()
    figsize : tuple
        Figure size
    output_path : str
        Optional path to save figure
    
    Returns:
    --------
    None (displays plots)
    """
    
    x_label_map = {0: "No eQTL", 1: "Has eQTL"}

    n_comparisons = len(contingency_results)
    fig, axes = plt.subplots(1, n_comparisons, figsize=figsize)
    
    if n_comparisons == 1:
        axes = [axes]
    
    for idx, (comp_name, result) in enumerate(contingency_results.items()):
        contingency = result["contingency_table"]
        chi2 = result["chi2_statistic"]
        pval = result["p_value"]
        labels = result["labels"]
        
        # Compute row-normalized percentages
        contingency_pct = contingency.div(contingency.sum(axis=1), axis=0) * 100
        
        ax = axes[idx]
        
        sns.heatmap(
            contingency_pct,
            annot=False,
            cmap="YlOrRd",
            cbar_kws={"label": "% within group"},
            ax=ax,
            vmin=0,
            vmax=100,
            linewidths=0.5,
            linecolor="white",
        )
        
        # Annotate each cell with count and percentage, centered
        for i in range(contingency.shape[0]):
            for j in range(contingency.shape[1]):
                count = contingency.iloc[i, j]
                pct = contingency_pct.iloc[i, j]
                ax.text(j + 0.5, i + 0.5, f"{int(count):,}\n({pct:.1f}%)",
                        ha="center", va="center", fontsize=9,
                        color="black", fontweight="bold")
        
        ax.set_title(f"{comp_name}\nχ² = {chi2:.3f}, p = {pval:.2e}",
                     fontsize=11, fontweight="bold", pad=10)
        ax.set_xlabel("eQTL Match", labelpad=10)
        ax.set_ylabel("SNP Category", labelpad=8)
        
        ax.set_xticklabels(
            [x_label_map.get(c, str(c)) for c in contingency.columns],
            rotation=45, ha="right", fontsize=10
        )
        ax.set_yticklabels(labels[:contingency.shape[0]], rotation=0, fontsize=10)
    
    fig.tight_layout(pad=2.0)
    
    if output_path:
        plt.savefig(output_path, dpi=300, bbox_inches="tight")
        print(f"Figure saved to: {output_path}")
    
    plt.show()


### Run catalog SNP eQTL overlap analysis

In [ ]:
# Execute catalog SNP analysis
catalog_path = "/home/labs/davidgo/Collaboration/Tomas_Apes_Catalog_parsing_6/FILTER_INCLUSIVE_STRICT_6/INDEL_filtered/catalog.bed"

print("="*80)
print("Catalog SNP Analysis: eQTL Overlap Stratified by Oligo Activity")
print("="*80)

# Step 1: Load and classify catalog SNPs (with upfront sampling + subsampling of outside SNPs)
print("\n[Step 1] Loading catalog and classifying SNPs by oligo membership...")
catalog_snps = load_and_classify_catalog_snps(
    catalog_path,
    oligos,
    oligo_chr_col="chromosome",
    oligo_start_col="start",
    oligo_end_col="end",
    oligo_activity_col="differential_activity",
    catalog_sample_fraction=0.1,   # sample 10% of catalog upfront before any processing
    subsample_outside_snps=True,
    outside_snp_fraction=0.1,
    random_seed=42
)

# Print SNP group distributions
print(f"\nSNP group distribution:")
snp_group_counts = catalog_snps["snp_group"].value_counts()
print(snp_group_counts)
print(f"\nTotal SNPs (after subsampling): {len(catalog_snps)}")

# Step 2: Check eQTL overlap for each SNP
print(f"\n[Step 2] Checking eQTL overlap for catalog SNPs...")
catalog_snps = check_snp_eqtl_overlap(
    catalog_snps,
    eqtls_to_use,
    catalog_pos_col="start",
    catalog_chr_col="chromosome",
    eqtl_chr_col="chromosome",
    eqtl_pos_col="position"
)

# Print eQTL match distribution
print(f"\neQTL match distribution:")
print(catalog_snps["has_eqtl_match"].value_counts())
eqtl_match_rate = catalog_snps["has_eqtl_match"].sum() / len(catalog_snps) * 100
print(f"Overall eQTL match rate: {eqtl_match_rate:.2f}%")

# Step 3: Compute contingency tables
print(f"\n[Step 3] Computing contingency tables and chi-square tests...")
catalog_contingency_results = compute_catalog_contingency_tables(
    catalog_snps,
    bonferroni_correction=False
)

# Step 4: Visualize
print(f"\n[Step 4] Creating heatmap visualizations...")
plot_catalog_contingency_heatmaps(
    catalog_contingency_results,
    figsize=(15, 5),
    output_path=f"{output_path}/catalog_SNPs_eQTL_overlap_heatmaps.png"
)

print("\nCatalog SNP analysis complete!")


### Load GWAS catalog data

In [ ]:
gwas_df = pd.read_csv('/home/labs/davidgo/Collaboration/GenomeAnnotation/Human/GWAS_catalog/current/gwas-catalog-download-associations-alt-full.tsv', sep='\t')
gwas_phenotype_annotation = pd.read_csv('/home/labs/davidgo/Collaboration/GenomeAnnotation/Human/GWAS_catalog/current/gwas_catalog_trait-mappings_r2026-03-17.tsv', sep='\t') 
gwas_df = gwas_df.merge(gwas_phenotype_annotation, left_on='DISEASE/TRAIT', right_on='Disease trait', how='left')
print(gwas_df["Parent term"].value_counts(dropna=False))

### Reload oligo data

In [ ]:
# Read only the specified columns from the CSV file
oligos = pd.read_csv(f'humanMPRA/top_candidates/chondrocytes/humanMPRA_annotations_v3.csv', 
                     #usecols=range(0, 25), 
                     header=0)
print('Number of oligos in hMPRA:', len(oligos))
oligos = oligos.drop_duplicates(subset=["oligo"], keep = "first") #There are several HH oligos which are duplicated
print('Number of oligos in hMPRA without duplicates:', len(oligos))

min_DNA_counts = 50

oligos = oligos[(oligos['DNA_counts_raw_derived']>min_DNA_counts) &
                               (oligos['DNA_counts_raw_ancestral']>min_DNA_counts)]
print('Number of oligos in hMPRA after filtering for at least 50 DNA counts in both archaic and derived:', len(oligos))


### Apply ATAC-seq filter to oligos

In [ ]:

if ATAC_seq_filter:
    print('Number of oligos in hMPRA before filtering for being in an ATAC-seq peak:', len(oligos))
    oligos["ATAC_seq_bool"] = oligos["ATACseq_peaks_fetal_chondrocytes"].abs() < ATAC_seq_distance
    oligos = oligos[oligos["ATAC_seq_bool"] == True]
    print('Number of oligos in hMPRA after filtering for being in an ATAC-seq peak:', len(oligos))


## Distance to Closest eQTL: Chondrocyte vs GTEx

In [ ]:
gtex_path = "/home/labs/davidgo/Collaboration/GenomeAnnotation/Human/GTEx/GTEx_Analysis_v7_eQTL/combined_signif_files.txt"

n_chond = len(cartilage_eQTLs)
sample_fraction = 0.05  # read 5% of rows per chunk; increase if ValueError raised below

rng = np.random.RandomState(42)
chunks = []
for chunk in pd.read_csv(gtex_path, sep="\t", usecols=["chr", "var_end"], chunksize=100_000):
    n_keep = max(1, int(len(chunk) * sample_fraction))
    chunks.append(chunk.sample(n=n_keep, random_state=rng.randint(0, 2**31)))

gtex_eQTLs = (pd.concat(chunks)
               .rename(columns={"chr": "chromosome", "var_end": "position"})
               .drop_duplicates(subset=["chromosome", "position"])
               .reset_index(drop=True))
gtex_eQTLs["chromosome"] = gtex_eQTLs["chromosome"].astype(str)

if len(gtex_eQTLs) < n_chond:
    raise ValueError(
        f"Only {len(gtex_eQTLs)} unique GTEx positions after {sample_fraction*100:.0f}% sampling "
        f"— need {n_chond}. Increase sample_fraction."
    )

gtex_eQTLs_sampled = gtex_eQTLs.sample(n=n_chond, random_state=42).reset_index(drop=True)

if len(gtex_eQTLs_sampled) != n_chond:
    raise ValueError(f"Subsampled GTEx size {len(gtex_eQTLs_sampled)} != chond size {n_chond}")

print(f"Chondrocyte eQTLs: {n_chond}")
print(f"GTEx eQTLs (subsampled): {len(gtex_eQTLs_sampled)}")

In [ ]:
gtex_eQTLs_sampled

In [ ]:
def compute_min_distances(variants_df, eqtl_df,
                          var_chr_col="chromosome",
                          var_start_col="start",
                          var_end_col="end",
                          eqtl_chr_col="chromosome",
                          eqtl_pos_col="position"):
    """
    For each variant interval, compute minimum distance to nearest eQTL on the same chromosome.
    Distance = 0 if eQTL falls within [start, end]. For point variants, set start==end.
    """
    variants = variants_df.copy()
    variants["_chrom"] = variants[var_chr_col].astype(str).str.replace("^chr", "", regex=True)

    eqtl_proc = eqtl_df[[eqtl_chr_col, eqtl_pos_col]].dropna().copy()
    eqtl_proc["_chrom"] = eqtl_proc[eqtl_chr_col].astype(str).str.replace("^chr", "", regex=True)
    eqtl_proc[eqtl_pos_col] = eqtl_proc[eqtl_pos_col].astype(int)

    eqtl_by_chrom = {}
    for chrom, grp in eqtl_proc.groupby("_chrom"):
        eqtl_by_chrom[chrom] = np.sort(grp[eqtl_pos_col].values)

    distances = np.full(len(variants), np.nan)
    for i, (_, row) in enumerate(variants.iterrows()):
        chrom = row["_chrom"]
        if chrom not in eqtl_by_chrom:
            continue
        arr = eqtl_by_chrom[chrom]
        start = int(row[var_start_col])
        end = int(row[var_end_col])
        # Binary search for nearest eQTL
        idx = np.searchsorted(arr, start)
        candidates = []
        for j in [idx - 1, idx]:
            if 0 <= j < len(arr):
                pos = arr[j]
                if start <= pos <= end:
                    candidates.append(0)
                else:
                    candidates.append(min(abs(pos - start), abs(pos - end)))
        if candidates:
            distances[i] = min(candidates)

    return distances

In [ ]:
# Extract oligo groups
oligos_proc = oligos.copy()
oligos_proc["_act"] = oligos_proc["differential_activity"].map(
    {True: "diff_active", False: "active_not_diff",
     1: "diff_active", 0: "active_not_diff"}
)
oligos_proc.loc[oligos_proc["_act"].isna(), "_act"] = "not_active"

groups = {
    "diff_active":     oligos_proc[oligos_proc["_act"] == "diff_active"],
    "active_not_diff": oligos_proc[oligos_proc["_act"] == "active_not_diff"],
    "not_active":      oligos_proc[oligos_proc["_act"] == "not_active"],
}

# "not in oligo" group from previously computed catalog_snps
outside_snps = catalog_snps[catalog_snps["snp_group"] == "outside"].copy()
outside_snps["start_1based"] = outside_snps["start"] + 1  # BED 0-based → 1-based
groups["not_in_oligo"] = outside_snps.rename(columns={"chromosome": "chromosome"})

# Compute distances (use start_1based==end for point SNPs in outside group)
distance_results = {}
for group_name, df in groups.items():
    if group_name == "not_in_oligo":
        df = df.copy()
        df["_end"] = df["start_1based"]
        chond_dists = compute_min_distances(df, cartilage_eQTLs,
            var_start_col="start_1based", var_end_col="_end")
        gtex_dists  = compute_min_distances(df, gtex_eQTLs_sampled,
            var_start_col="start_1based", var_end_col="_end")
    else:
        chond_dists = compute_min_distances(df, cartilage_eQTLs)
        gtex_dists  = compute_min_distances(df, gtex_eQTLs_sampled)
    
    distance_results[group_name] = {
        "chond": chond_dists[~np.isnan(chond_dists)],
        "gtex":  gtex_dists[~np.isnan(gtex_dists)],
    }
    print(f"{group_name}: {len(distance_results[group_name]['chond'])} variants with distances")

In [ ]:
from scipy.stats import ks_2samp

fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=True)
group_order = ["diff_active", "active_not_diff", "not_active", "not_in_oligo"]
group_labels = ["Diff. active", "Active (not diff.)", "Not active", "Not in oligo"]

for ax, group_name, group_label in zip(axes, group_order, group_labels):
    chond = distance_results[group_name]["chond"]
    gtex  = distance_results[group_name]["gtex"]

    # ECDF
    for arr, label, color in [(chond, "Chondrocyte", "steelblue"), (gtex, "GTEx (other)", "tomato")]:
        sorted_arr = np.sort(arr)
        ecdf_y = np.arange(1, len(sorted_arr) + 1) / len(sorted_arr)
        ax.plot(sorted_arr, ecdf_y, label=label, color=color)

    ks_stat, ks_pval = ks_2samp(chond, gtex)
    ax.set_title(f"{group_label}\nKS p = {ks_pval:.2e}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Distance to nearest eQTL (bp)")
    ax.set_xscale("symlog", linthresh=100)
    ax.legend(fontsize=9)

axes[0].set_ylabel("ECDF")
fig.suptitle("Distance to Nearest eQTL: Chondrocyte vs GTEx (subsampled equal N)", fontweight="bold")
fig.tight_layout()
plt.savefig(f"{output_path}/eQTL_distance_ECDF_chond_vs_GTEx.png", dpi=300, bbox_inches="tight")
plt.show()

# Do the differentially active oligos overlap with cartilage eQTLs?

#### Read eQTL data

In [ ]:
high_grade_cartilage_eQTLs = pd.read_csv("/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Expression/Cartilage/Human/cartilage_eQTLs_for_healthy_and_OA_individuals/processed_data/HighGradeCartilage_significant_eQTLs.txt", sep="\t")
low_grade_cartilage_eQTLs = pd.read_csv("/home/labs/davidgo/Collaboration/USEFUL_DATASETS/Expression/Cartilage/Human/cartilage_eQTLs_for_healthy_and_OA_individuals/processed_data/LowGradeCartilage_significant_eQTLs.txt", sep="\t")

high_grade_cartilage_eQTLs["origin"] = "high_grade"
low_grade_cartilage_eQTLs["origin"] = "low_grade"

cartilage_eQTLs = pd.concat(
    [high_grade_cartilage_eQTLs, low_grade_cartilage_eQTLs],
    ignore_index=True
)

In [ ]:
cartilage_eQTLs["chromosome"] = cartilage_eQTLs["genotype_id"].apply(lambda x: x.split(":")[0])
cartilage_eQTLs["position"] = cartilage_eQTLs["genotype_id"].apply(lambda x: int(x.split(":")[1]))

In [ ]:
cartilage_eQTLs

#### Read hMPRA data

In [ ]:
# Read only the specified columns from the CSV file
oligos = pd.read_csv(f'/home/labs/davidgo/Collaboration/humanMPRA/top_candidates/chondrocytes/humanMPRA_annotations_v3.csv', 
                     #usecols=range(0, 25), 
                     header=0)
print('Number of oligos in hMPRA:', len(oligos))
oligos = oligos.drop_duplicates(subset=["oligo"], keep = "first") #There are several HH oligos which are duplicated
print('Number of oligos in hMPRA without duplicates:', len(oligos))

min_DNA_counts = 50

oligos = oligos[(oligos['DNA_counts_raw_derived']>min_DNA_counts) &
                               (oligos['DNA_counts_raw_ancestral']>min_DNA_counts)]
print('Number of oligos in hMPRA after filtering for at least 50 DNA counts in both archaic and derived:', len(oligos))


ATAC_seq_filter = True
ATAC_seq_distance = 5000

if ATAC_seq_filter:
    print('Number of oligos in hMPRA before filtering for being in an ATAC-seq peak:', len(oligos))
    oligos["ATAC_seq_bool"] = oligos["ATACseq_peaks_fetal_chondrocytes"].abs() < ATAC_seq_distance
    oligos = oligos[oligos["ATAC_seq_bool"] == True]
    print('Number of oligos in hMPRA after filtering for being in an ATAC-seq peak:', len(oligos))


In [ ]:
# Define the distance calculation function for eQTLs
def calculate_eqtl_distances(oligos_df, eqtl_df,
                            oligo_chr_col="chromosome",
                            oligo_start_col="start",
                            oligo_end_col="end",
                            oligo_activity_col="differential_activity",
                            eqtl_chr_col="chromosome",
                            eqtl_pos_col="position",
                            output_dist_col="nearest_eqtl_distance_bp",
                            activity_type="diff_vs_active",
                            group_labels=None):
    """
    Calculate distance from each oligo to the nearest eQTL.
    
    Parameters:
    -----------
    oligos_df : DataFrame
        Oligo dataframe with coordinates and activity labels
    eqtl_df : DataFrame
        eQTL dataframe with SNP positions
    oligo_chr_col : str
        Chromosome column name in oligos_df
    oligo_start_col : str
        Start coordinate column in oligos_df
    oligo_end_col : str
        End coordinate column in oligos_df
    oligo_activity_col : str
        Differential activity column in oligos_df
    eqtl_chr_col : str
        Chromosome column name in eqtl_df
    eqtl_pos_col : str
        Position column in eqtl_df
    output_dist_col : str
        Name for output distance column
    activity_type : str
        Type of comparison: "diff_vs_active" (default) compares differentially active vs active but not differentially active.
        "active_vs_non_active" compares all active oligos vs non-active oligos (with NA values).
    group_labels : dict
        Dict mapping group values to labels. If None, uses defaults based on activity_type.
    
    Returns:
    --------
    DataFrame with distances and activity group labels
    """
    
    if activity_type not in ["diff_vs_active", "active_vs_non_active"]:
        raise ValueError(f"activity_type must be 'diff_vs_active' or 'active_vs_non_active', got {activity_type}")
    
    # Prepare oligo data
    oligos_dist = oligos_df.copy()
    cols_needed = [oligo_chr_col, oligo_start_col, oligo_end_col, oligo_activity_col]
    missing = [c for c in cols_needed if c not in oligos_dist.columns]
    if missing:
        raise ValueError(f"Missing required oligos columns: {missing}")
    
    # For coordinates, we need chr/start/end to be non-null
    oligos_dist = oligos_dist[[oligo_chr_col, oligo_start_col, oligo_end_col, oligo_activity_col]].copy()
    oligos_dist[oligo_start_col] = pd.to_numeric(oligos_dist[oligo_start_col], errors="coerce")
    oligos_dist[oligo_end_col] = pd.to_numeric(oligos_dist[oligo_end_col], errors="coerce")
    oligos_dist = oligos_dist.dropna(subset=[oligo_chr_col, oligo_start_col, oligo_end_col]).copy()
    oligos_dist[oligo_start_col] = oligos_dist[oligo_start_col].astype(int)
    oligos_dist[oligo_end_col] = oligos_dist[oligo_end_col].astype(int)
    oligos_dist["chrom_clean"] = oligos_dist[oligo_chr_col].astype(str).str.replace("^chr", "", regex=True)
    
    # Prepare eQTL data
    eqtl_proc = eqtl_df.copy()
    missing_eqtl = [c for c in [eqtl_chr_col, eqtl_pos_col] if c not in eqtl_proc.columns]
    if missing_eqtl:
        raise ValueError(f"Missing required eQTL columns: {missing_eqtl}")
    
    eqtl_proc = eqtl_proc[[eqtl_chr_col, eqtl_pos_col]].dropna().copy()
    eqtl_proc[eqtl_pos_col] = pd.to_numeric(eqtl_proc[eqtl_pos_col], errors="coerce")
    eqtl_proc = eqtl_proc.dropna(subset=[eqtl_pos_col]).copy()
    eqtl_proc[eqtl_pos_col] = eqtl_proc[eqtl_pos_col].astype(int)
    eqtl_proc["chrom_clean"] = eqtl_proc[eqtl_chr_col].astype(str).str.replace("^chr", "", regex=True)
    
    # Compute distance from oligo interval (distance = 0 if eQTL falls within oligo)
    nearest_distances = []
    for chrom, grp in oligos_dist.groupby("chrom_clean"):
        eqtl_pos = np.sort(eqtl_proc.loc[eqtl_proc["chrom_clean"] == chrom, eqtl_pos_col].values)
        
        starts = grp[oligo_start_col].to_numpy()
        ends = grp[oligo_end_col].to_numpy()

        if len(eqtl_pos) == 0:
            nearest = np.full(len(starts), np.nan)
        else:
            nearest = np.zeros(len(starts))
            
            for i in range(len(starts)):
                start = starts[i]
                end = ends[i]
                
                eqtls_inside = np.sum((eqtl_pos >= start) & (eqtl_pos <= end))
                if eqtls_inside > 0:
                    nearest[i] = 0
                else:
                    idx_left = np.searchsorted(eqtl_pos, start, side='right') - 1
                    idx_right = np.searchsorted(eqtl_pos, end, side='left')
                    
                    left_dist = np.inf
                    right_dist = np.inf
                    
                    if idx_left >= 0:
                        left_dist = start - eqtl_pos[idx_left]
                    if idx_right < len(eqtl_pos):
                        right_dist = eqtl_pos[idx_right] - end
                    
                    nearest[i] = min(left_dist, right_dist)

        out = grp.copy()
        out[output_dist_col] = nearest
        nearest_distances.append(out)

    result = pd.concat(nearest_distances, ignore_index=True)
    result = result.dropna(subset=[output_dist_col]).copy()

    # Handle activity classification based on activity_type
    if activity_type == "diff_vs_active":
        # Compare differentially active vs active but not differentially active
        # Keep only rows with explicit TRUE/FALSE values
        status_map = {
            True: True, False: False,
            "TRUE": True, "FALSE": False,
            "True": True, "False": False,
            "true": True, "false": False,
            1: True, 0: False
        }
        result["diff_bool"] = result[oligo_activity_col].map(status_map)
        result = result[result["diff_bool"].isin([True, False])].copy()

        # Create group labels
        if group_labels is None:
            group_labels = {True: "Differentially active", False: "Active but not differentially active"}
        
        result["group"] = result["diff_bool"].map(group_labels)

    elif activity_type == "active_vs_non_active":
        # Compare non-NaN values (active) vs NaN values (non-active)
        # Keep both groups - do NOT filter out rows since NaN distinction is important
        result["is_nan"] = result[oligo_activity_col].isna()
        
        # Create group labels
        if group_labels is None:
            group_labels = {False: "Active", True: "Non-active"}  # False = not NaN = active; True = NaN = non-active
        
        result["group"] = result["is_nan"].map(group_labels)
        
    return result


def plot_ecdf_with_ks_test(plot_df,
                          distance_col="nearest_eqtl_distance_bp",
                          group_col="group",
                          figsize=(9, 5),
                          title_suffix="",
                          max_distance=10000,
                          downsample=False):
    """
    Plot ECDF of SNP distances and perform Kolmogorov-Smirnov test.
    
    Parameters:
    -----------
    plot_df : DataFrame
        DataFrame with distance and group columns (output from calculate_eqtl_distances)
    distance_col : str
        Name of distance column
    group_col : str
        Name of group/category column
    figsize : tuple
        Figure size
    title_suffix : str
        Suffix for figure title
    max_distance : int
        Maximum distance in base pairs to include in analysis (default: 10000).
        Values above this threshold will be filtered out.
    downsample : bool
        If True, downsample the larger group to match the size of the smaller group (default: False).
    
    Returns:
    --------
    K-S test results as dictionary
    """
    
    # Filter data by max_distance
    plot_df_filtered = plot_df[plot_df[distance_col] <= max_distance].copy()
    
    # Downsample larger group if requested
    if downsample:
        group_sizes = plot_df_filtered[group_col].value_counts()
        min_size = group_sizes.min()
        
        # Downsample each group to min_size
        downsampled_groups = []
        for group_name in plot_df_filtered[group_col].unique():
            group_data = plot_df_filtered[plot_df_filtered[group_col] == group_name]
            if len(group_data) > min_size:
                group_data = group_data.sample(n=min_size, random_state=42)
            downsampled_groups.append(group_data)
        
        plot_df_filtered = pd.concat(downsampled_groups, ignore_index=True)
    
    # ECDF plot
    plt.figure(figsize=figsize)

    for group, grp in plot_df_filtered.groupby(group_col):
        x = grp[distance_col].to_numpy(dtype=float)
        x = x[np.isfinite(x)]
        x = np.sort(x)

        if x.size == 0:
            continue

        y = np.arange(1, x.size + 1) / x.size

        # +1 avoids log(0) issues while preserving ordering
        plt.step(x + 1, y, where="post", label=group, linewidth=2)

    plt.xscale("log")
    plt.xlabel(f"Distance to closest eQTL\n(bp, +1 for log scale)")
    plt.ylabel("Cumulative Probability")
    plt.legend(title="Oligo group")
    if title_suffix:
        plt.title(f"eQTL Distance Distribution\n{title_suffix}")
    plt.tight_layout()
    plt.show()

    # Quick summary stats
    print("Summary statistics by group:")
    print(plot_df_filtered.groupby(group_col)[distance_col].describe())

    # Kolmogorov-Smirnov test
    print("\n" + "="*70)
    print("Kolmogorov-Smirnov Test: Compare eQTL distance distributions")
    print("="*70)

    groups = plot_df_filtered[group_col].unique()
    ks_results = {}
    
    if len(groups) == 2:
        group1_data = plot_df_filtered[plot_df_filtered[group_col] == groups[0]][distance_col].dropna().values
        group2_data = plot_df_filtered[plot_df_filtered[group_col] == groups[1]][distance_col].dropna().values
        
        ks_stat, ks_pvalue = stats.ks_2samp(group1_data, group2_data)
        
        print(f"\nGroup 1: {groups[0]}")
        print(f"  N = {len(group1_data)}, Mean = {np.mean(group1_data):.1f} bp, Median = {np.median(group1_data):.1f} bp")
        print(f"\nGroup 2: {groups[1]}")
        print(f"  N = {len(group2_data)}, Mean = {np.mean(group2_data):.1f} bp, Median = {np.median(group2_data):.1f} bp")
        print(f"\nKolmogorov-Smirnov Test:")
        print(f"  Test statistic = {ks_stat:.4f}")
        print(f"  P-value = {ks_pvalue:.4e}")
        
        if ks_pvalue < 0.05:
            print(f"  ✓ Significant difference (p < 0.05): Distributions are significantly different")
        else:
            print(f"  ✗ No significant difference (p >= 0.05): Distributions are not significantly different")
        
        ks_results = {
            "group1": groups[0],
            "group2": groups[1],
            "n_group1": len(group1_data),
            "n_group2": len(group2_data),
            "mean_group1": np.mean(group1_data),
            "mean_group2": np.mean(group2_data),
            "median_group1": np.median(group1_data),
            "median_group2": np.median(group2_data),
            "ks_statistic": ks_stat,
            "ks_pvalue": ks_pvalue,
            "significant": ks_pvalue < 0.05
        }
    else:
        print(f"Warning: Expected 2 groups, found {len(groups)}")
    
    return ks_results


# Calculate distances from oligos to cartilage eQTLs (diff vs active comparison)
print("Analyzing distances between oligos and cartilage eQTLs...")
print(f"Number of cartilage eQTLs: {len(cartilage_eQTLs)}")

eqtl_plot_df = calculate_eqtl_distances(
    oligos,
    cartilage_eQTLs,
    oligo_chr_col="chromosome",
    oligo_start_col="start",
    oligo_end_col="end",
    oligo_activity_col="differential_activity",
    eqtl_chr_col="chromosome",
    eqtl_pos_col="position",
    output_dist_col="nearest_eqtl_distance_bp",
    activity_type="diff_vs_active"
)

print(f"Oligos with eQTL distance data (diff_vs_active): {len(eqtl_plot_df)}")

# Create ECDF plot and run K-S test for cartilage eQTLs
eqtl_ks_results = plot_ecdf_with_ks_test(
    eqtl_plot_df,
    distance_col="nearest_eqtl_distance_bp",
    group_col="group",
    title_suffix="(Cartilage - Differentially Active vs Active)",downsample=False
)

In [ ]:
# Now compare all active oligos vs non-active oligos
print("\n" + "="*80)
print("Comparing all active oligos vs non-active oligos...")
print("="*80)

eqtl_plot_df_active_vs_non = calculate_eqtl_distances(
    oligos,
    cartilage_eQTLs,
    oligo_chr_col="chromosome",
    oligo_start_col="start",
    oligo_end_col="end",
    oligo_activity_col="differential_activity",
    eqtl_chr_col="chromosome",
    eqtl_pos_col="position",
    output_dist_col="nearest_eqtl_distance_bp",
    activity_type="active_vs_non_active"
)

print(f"Oligos with eQTL distance data (active_vs_non_active): {len(eqtl_plot_df_active_vs_non)}")

# Create ECDF plot and run K-S test for active vs non-active
eqtl_ks_results_active_vs_non = plot_ecdf_with_ks_test(
    eqtl_plot_df_active_vs_non,
    distance_col="nearest_eqtl_distance_bp",
    group_col="group",
    title_suffix="(Cartilage - Active vs Non-active)",
    max_distance=10000
)

In [ ]:
# Create ECDF plot and run K-S test for cartilage eQTLs
# Calculate distances from oligos to cartilage eQTLs
print("Analyzing distances between oligos and cartilage eQTLs...")
print(f"Number of cartilage eQTLs: {len(cartilage_eQTLs)}")

eqtl_plot_df = calculate_eqtl_distances(
    oligos,
    cartilage_eQTLs,
    oligo_chr_col="chromosome",
    oligo_start_col="start",
    oligo_end_col="end",
    oligo_activity_col="differential_activity",
    eqtl_chr_col="chromosome",
    eqtl_pos_col="position",
    output_dist_col="nearest_eqtl_distance_bp"
)

print(f"Oligos with eQTL distance data: {len(eqtl_plot_df)}")

eqtl_ks_results = plot_ecdf_with_ks_test(
    eqtl_plot_df,
    distance_col="nearest_eqtl_distance_bp",
    group_col="group",
    title_suffix="(Cartilage)",
    max_distance=10000
)